# SWAN-SF - Build 3D Tensors for Partitions 1-5

This notebook rebuilds **Partition 1** and builds the same 3D tensor format for **Partitions 2-5**.

It expects your Google Drive archives to follow this naming pattern:

```text
/content/drive/MyDrive/solar_flare_forecasting/Data/partition1_instances.tar.gz
/content/drive/MyDrive/solar_flare_forecasting/Data/partition2_instances.tar.gz
/content/drive/MyDrive/solar_flare_forecasting/Data/partition3_instances.tar.gz
/content/drive/MyDrive/solar_flare_forecasting/Data/partition4_instances.tar.gz
/content/drive/MyDrive/solar_flare_forecasting/Data/partition5_instances.tar.gz
```

Each output tensor has the shape:

```text
(N files, 60 timesteps, F features)
```

where `F` is detected from the files after dropping the 8 sparse JSON `_LABEL` columns and adding a row-level `was_interpolated` flag.

## What this notebook produces

For each partition, the notebook saves:

| Output | Purpose |
|---|---|
| `partition{p}_combined_clean.npz` | Main modeling tensor, excluding files with 5+ interpolated rows |
| `partition{p}_metadata_clean.csv` | Metadata for the clean tensor rows |
| `partition{p}_metadata_all.csv` | Metadata for every successfully processed file, including high-imputation files |
| `partition{p}_clean_indices.npy` | Row indices in the all-file metadata that pass the clean rule |
| `partition{p}_feature_columns.json` | Feature order used along axis 2 of the tensor |
| `partition{p}_skipped_files.csv` | Files skipped due to unexpected row count, columns, or residual NaNs |
| `swan_sf_tensor_build_summary.csv` | One-row-per-partition summary |
| `swan_sf_tensor_manifest.csv` | Saved file paths plus the train/test partition split |

By default, the notebook saves the **clean tensor** only. It also has an option to save the full all-files sensitivity tensor, but that is disabled by default to avoid duplicating very large arrays.

## Important modeling note

This notebook **does not normalize** the tensors. That is intentional.

For your split:

```text
Train: partitions 1, 2, 3, 5
Test:  partition 4
```

normalization should be fit using **training partitions only**, then applied to partition 4. Fitting normalization on all five partitions would leak test-set information into training.

## 1. Imports

In [ ]:
import os
import re
import json
import gc
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 100)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration

You should only need to edit this section if your Drive folders are different.

`PARTITIONS_TO_BUILD = [1, 2, 3, 4, 5]` rebuilds all partitions. If you want to test the notebook quickly first, temporarily set it to `[1]`.

In [ ]:
# -----------------------------
# Main paths
# -----------------------------
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/solar_flare_forecasting/Data')
OUTPUT_DIR = Path('/content/drive/MyDrive/solar_flare_forecasting/processed_tensors')
LOCAL_EXTRACT_ROOT = Path('/content/solar_flare_data_extracted')

# -----------------------------
# Which partitions to build
# -----------------------------
PARTITIONS_TO_BUILD = [1, 2, 3, 4, 5]

# split for later modeling
TRAIN_PARTITIONS = [1, 2, 3, 5]
TEST_PARTITIONS = [4]

# -----------------------------
# Shape / cleaning policy
# -----------------------------
EXPECTED_ROWS = 60
MAX_INTERPOLATED_ROWS_FOR_CLEAN = 4   # clean set keeps 0-4; excludes 5+
TENSOR_DTYPE = np.float32

# -----------------------------
# Saving options
# -----------------------------
SAVE_CLEAN_TENSOR = True              # recommended main modeling tensor
SAVE_ALL_SENSITIVITY_TENSOR = False   # set True only if you want a full all-file tensor copy too

# -----------------------------
# Extraction behavior
# -----------------------------
FORCE_REEXTRACT = True                # True guarantees Partition 1 is rebuilt from scratch
DELETE_EXTRACTED_AFTER_EACH_PARTITION = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

print('Drive data folder:', DRIVE_DATA_DIR)
print('Output folder:    ', OUTPUT_DIR)
print('Local extract dir:', LOCAL_EXTRACT_ROOT)

## 4. Constants and column policy

Main choices:

- Drop the 8 sparse JSON `_LABEL` columns.
- Interpolate missing smooth numeric feature values **within each file only**.
- Keep the fixed 60-row structure.
- Add `was_interpolated` as a row-level feature.
- Track high-imputation files with metadata instead of deleting rows.

In [ ]:
LABEL_COLS = [
    'BFLARE_LABEL', 'CFLARE_LABEL', 'MFLARE_LABEL', 'XFLARE_LABEL',
    'BFLARE_LABEL_LOC', 'CFLARE_LABEL_LOC', 'MFLARE_LABEL_LOC', 'XFLARE_LABEL_LOC',
]

# These are not smooth magnetic-field quantities, so do not linearly interpolate them.
# SPEI is retained as its own feature and filled with a sentinel only if it is missing.
NO_INTERPOLATE = ['XR_MAX', 'XRQUALITY', 'IS_TMFI', 'QUALITY', 'SPEI']

QUALITY_SENTINEL_FOR_MISSING = -1
FLAG_SENTINEL_FOR_MISSING = -1

TRUE_TOKENS = {True, 'True', 'TRUE', 'true', 'T', 't', 1, '1'}
FALSE_TOKENS = {False, 'False', 'FALSE', 'false', 'F', 'f', 0, '0'}

# The first successfully processed file locks the canonical feature order.
column_reference = {'numeric_cols': None}

## 5. Helper functions - extraction and file discovery

In [ ]:
def archive_path_for_partition(partition: int) -> Path:
    return DRIVE_DATA_DIR / f'partition{partition}_instances.tar.gz'


def extract_partition_archive(partition: int) -> Path:
    """Extract one partition archive to local Colab SSD and return its local root."""
    archive_path = archive_path_for_partition(partition)
    extract_dir = LOCAL_EXTRACT_ROOT / f'partition{partition}'

    if not archive_path.exists():
        raise FileNotFoundError(f'Archive not found: {archive_path}')

    if FORCE_REEXTRACT and extract_dir.exists():
        shutil.rmtree(extract_dir)

    if extract_dir.exists() and any(extract_dir.iterdir()):
        print(f'Partition {partition}: using existing extraction at {extract_dir}')
        return extract_dir

    extract_dir.mkdir(parents=True, exist_ok=True)
    print(f'Partition {partition}: extracting {archive_path.name} ...')

    # Use system tar for speed in Colab. subprocess list form handles spaces safely.
    result = subprocess.run(
        ['tar', '-xzf', str(archive_path), '-C', str(extract_dir)],
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f'tar failed for partition {partition} with code {result.returncode}\n'
            f'STDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}'
        )

    print(f'Partition {partition}: extraction complete.')
    return extract_dir


def find_label_dirs(root: Path, folder_name: str):
    """Find directories literally named FL or NF, regardless of nesting depth."""
    root = Path(root)
    return sorted([Path(d) for d, _, _ in os.walk(root) if Path(d).name == folder_name])


def discover_instance_files(extract_dir: Path):
    """Return a sorted list of (filepath, label) pairs for all FL/NF instance files."""
    extract_dir = Path(extract_dir)
    file_list = []

    for label in ['FL', 'NF']:
        for label_dir in find_label_dirs(extract_dir, label):
            for path in sorted(label_dir.iterdir()):
                if path.is_file() and not path.name.startswith('.'):
                    file_list.append((path, label))

    file_list = sorted(file_list, key=lambda x: (x[1], str(x[0])))
    return file_list

## 6. Helper functions - cleaning and metadata

In [ ]:
def coerce_known_booleans(df: pd.DataFrame) -> pd.DataFrame:
    """Convert True/False-style columns to 0/1 so they can be included in the tensor."""
    for col in df.columns:
        values = set(df[col].dropna().unique().tolist())
        if values and values.issubset(TRUE_TOKENS | FALSE_TOKENS):
            df[col] = df[col].apply(
                lambda v: 1 if v in TRUE_TOKENS else (0 if v in FALSE_TOKENS else np.nan)
            )
    return df


def classify_columns(df: pd.DataFrame):
    """Split columns into numeric tensor columns and metadata/string columns."""
    numeric_cols, metadata_cols = [], []
    for col in df.columns:
        try:
            pd.to_numeric(df[col])
            numeric_cols.append(col)
        except (ValueError, TypeError):
            metadata_cols.append(col)
    return numeric_cols, metadata_cols


def count_leading_true(mask) -> int:
    count = 0
    for value in mask:
        if bool(value):
            count += 1
        else:
            break
    return count


def count_trailing_true(mask) -> int:
    count = 0
    for value in reversed(mask):
        if bool(value):
            count += 1
        else:
            break
    return count


def parse_filename(filepath: Path, fl_nf_label: str):
    """Extract useful metadata from SWAN-SF instance filenames when available."""
    fname = Path(filepath).name

    flux_match = re.match(r'^([A-Za-z0-9.]+)@', fname)
    flare_class = flux_match.group(1) if flux_match else fl_nf_label

    harp_match = re.search(r'ar(\d+)', fname)
    harpnum = int(harp_match.group(1)) if harp_match else -1

    start_match = re.search(r'_s([^_]+)', fname)
    end_match = re.search(r'_e([^_]+)', fname)
    start_time = start_match.group(1) if start_match else ''
    end_time = end_match.group(1) if end_match else ''

    return {
        'source_file': fname,
        'flare_class': flare_class,
        'harpnum': harpnum,
        'start_time': start_time,
        'end_time': end_time,
        'fl_nf_label': fl_nf_label,
        'y_flare': 1 if fl_nf_label == 'FL' else 0,
    }

## 7. Per-file processing function

In [ ]:
def process_file(filepath: Path, fl_nf_label: str, partition: int, extract_dir: Path):
    """Read one instance file and return (60, F) feature matrix plus metadata."""
    filepath = Path(filepath)

    # The instance files are tab-separated in the SWAN-SF partition archives.
    df = pd.read_csv(filepath, sep='\t')
    df = df.drop(columns=[c for c in LABEL_COLS if c in df.columns])
    df = coerce_known_booleans(df)

    if len(df) != EXPECTED_ROWS:
        return None, {'source_file': filepath.name, 'reason': f'unexpected row count: {len(df)}'}

    numeric_cols, metadata_cols = classify_columns(df)

    # Lock feature order from the first successfully processed file.
    if column_reference['numeric_cols'] is None:
        column_reference['numeric_cols'] = numeric_cols
        print(f'Canonical numeric feature count before was_interpolated: {len(numeric_cols)}')
    else:
        missing = [c for c in column_reference['numeric_cols'] if c not in numeric_cols]
        if missing:
            return None, {'source_file': filepath.name, 'reason': f'missing expected columns: {missing}'}
        numeric_cols = column_reference['numeric_cols']

    # Coerce all canonical tensor columns to numeric.
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Row-level missingness is based on interpolatable columns before filling.
    interp_cols = [c for c in numeric_cols if c not in NO_INTERPOLATE]
    was_interpolated = df[interp_cols].isna().any(axis=1).to_numpy()

    n_interpolated_rows = int(was_interpolated.sum())
    has_middle_interpolation = bool(was_interpolated[1:-1].any())
    has_boundary_interpolation = bool(was_interpolated[0] or was_interpolated[-1])
    leading_interpolated_run = count_leading_true(was_interpolated)
    trailing_interpolated_run = count_trailing_true(was_interpolated)

    # Per-file interpolation: never interpolate across source-file boundaries.
    df[interp_cols] = df[interp_cols].interpolate(method='linear', limit_direction='both')

    # Fill missing quality/flag columns with explicit sentinels if needed.
    if 'QUALITY' in df.columns:
        df['QUALITY'] = df['QUALITY'].fillna(QUALITY_SENTINEL_FOR_MISSING)
    for flag_col in ['IS_TMFI', 'SPEI', 'XRQUALITY']:
        if flag_col in df.columns:
            df[flag_col] = df[flag_col].fillna(FLAG_SENTINEL_FOR_MISSING)

    # Add row-level imputation flag as a model-visible feature.
    df['was_interpolated'] = was_interpolated.astype(int)
    final_cols = numeric_cols + ['was_interpolated']

    feature_matrix = df[final_cols].to_numpy(dtype=TENSOR_DTYPE)

    if np.isnan(feature_matrix).any():
        nan_per_col = np.isnan(feature_matrix).sum(axis=0)
        bad_cols = [final_cols[i] for i, count in enumerate(nan_per_col) if count > 0]
        return None, {
            'source_file': filepath.name,
            'reason': f'residual NaNs after interpolation/fill in columns: {bad_cols}',
        }

    meta = parse_filename(filepath, fl_nf_label)
    meta.update({
        'partition': partition,
        'relative_path': str(filepath.relative_to(extract_dir)),
        'n_interpolated_rows': n_interpolated_rows,
        'interpolation_rate': n_interpolated_rows / EXPECTED_ROWS,
        'has_interpolation': bool(n_interpolated_rows > 0),
        'has_middle_interpolation': has_middle_interpolation,
        'has_boundary_interpolation': has_boundary_interpolation,
        'start_interpolated': bool(was_interpolated[0]),
        'end_interpolated': bool(was_interpolated[-1]),
        'leading_interpolated_run': leading_interpolated_run,
        'trailing_interpolated_run': trailing_interpolated_run,
        'clean_keep': bool(n_interpolated_rows <= MAX_INTERPOLATED_ROWS_FOR_CLEAN),
    })

    # Extra instance-level quality diagnostics.
    if 'XRQUALITY' in df.columns:
        meta['xrquality_min'] = float(df['XRQUALITY'].min())
        meta['xrquality_degraded'] = bool((df['XRQUALITY'] < 12).any())
        meta['xrquality_zero_rows'] = int((df['XRQUALITY'] == 0).sum())
    else:
        meta['xrquality_min'] = np.nan
        meta['xrquality_degraded'] = False
        meta['xrquality_zero_rows'] = 0

    if 'QUALITY' in df.columns:
        meta['quality_nonzero_rows'] = int(((df['QUALITY'] != 0) & (df['QUALITY'] != QUALITY_SENTINEL_FOR_MISSING)).sum())
        meta['quality_missing_sentinel_rows'] = int((df['QUALITY'] == QUALITY_SENTINEL_FOR_MISSING).sum())
    else:
        meta['quality_nonzero_rows'] = 0
        meta['quality_missing_sentinel_rows'] = 0

    if 'IS_TMFI' in df.columns:
        meta['is_tmfi_false_rows'] = int((df['IS_TMFI'] == 0).sum())
    else:
        meta['is_tmfi_false_rows'] = 0

    if 'SPEI' in df.columns:
        meta['spei_false_rows'] = int((df['SPEI'] == 0).sum())
    else:
        meta['spei_false_rows'] = 0

    return feature_matrix, meta

## 8. Save helpers

In [ ]:
def save_npz_dataset(path: Path, X: np.ndarray, meta_df: pd.DataFrame):
    """Save tensor plus commonly used side-car arrays."""
    path = Path(path)
    np.savez_compressed(
        path,
        X=X,
        partition=meta_df['partition'].to_numpy(dtype=np.int16),
        y_flare=meta_df['y_flare'].to_numpy(dtype=np.int8),
        fl_nf_label=meta_df['fl_nf_label'].astype(str).to_numpy(),
        flare_class=meta_df['flare_class'].astype(str).to_numpy(),
        harpnum=meta_df['harpnum'].to_numpy(dtype=np.int32),
        source_file=meta_df['source_file'].astype(str).to_numpy(),
        clean_keep=meta_df['clean_keep'].to_numpy(dtype=bool),
        n_interpolated_rows=meta_df['n_interpolated_rows'].to_numpy(dtype=np.int16),
        has_boundary_interpolation=meta_df['has_boundary_interpolation'].to_numpy(dtype=bool),
        xrquality_degraded=meta_df['xrquality_degraded'].to_numpy(dtype=bool),
    )


def write_feature_columns(partition: int, feature_cols):
    out_path = OUTPUT_DIR / f'partition{partition}_feature_columns.json'
    with open(out_path, 'w') as f:
        json.dump(feature_cols, f, indent=2)
    return out_path

## 9. Build one partition

In [ ]:
def build_partition(partition: int):
    """Extract, process, save, and summarize one SWAN-SF partition."""
    extract_dir = extract_partition_archive(partition)
    file_list = discover_instance_files(extract_dir)

    n_fl = sum(1 for _, label in file_list if label == 'FL')
    n_nf = sum(1 for _, label in file_list if label == 'NF')
    print(f'Partition {partition}: found {len(file_list):,} instance files  (FL={n_fl:,}, NF={n_nf:,})')

    all_features = []
    records = []
    skipped = []

    for filepath, label in tqdm(file_list, desc=f'Processing partition {partition}'):
        try:
            matrix, result = process_file(filepath, label, partition, extract_dir)
        except Exception as e:
            skipped.append({'partition': partition, 'source_file': Path(filepath).name, 'reason': repr(e)})
            continue

        if matrix is None:
            result['partition'] = partition
            skipped.append(result)
            continue

        all_features.append(matrix)
        records.append(result)

    if not all_features:
        raise RuntimeError(f'Partition {partition}: no files were successfully processed.')

    X_all = np.stack(all_features).astype(TENSOR_DTYPE, copy=False)
    meta_all = pd.DataFrame(records)
    feature_cols = column_reference['numeric_cols'] + ['was_interpolated']

    # Sanity checks
    assert X_all.shape[0] == len(meta_all), 'Tensor and metadata row counts do not match.'
    assert X_all.shape[1] == EXPECTED_ROWS, f'Expected {EXPECTED_ROWS} timesteps, got {X_all.shape[1]}.'
    assert X_all.shape[2] == len(feature_cols), 'Feature-column count does not match tensor depth.'
    assert not np.isnan(X_all).any(), 'Unexpected NaNs survived processing.'

    clean_idx = np.flatnonzero(meta_all['clean_keep'].to_numpy())
    meta_clean = meta_all.iloc[clean_idx].reset_index(drop=True)

    # Paths
    clean_npz_path = OUTPUT_DIR / f'partition{partition}_combined_clean.npz'
    all_npz_path = OUTPUT_DIR / f'partition{partition}_combined_all_sensitivity.npz'
    meta_all_path = OUTPUT_DIR / f'partition{partition}_metadata_all.csv'
    meta_clean_path = OUTPUT_DIR / f'partition{partition}_metadata_clean.csv'
    clean_idx_path = OUTPUT_DIR / f'partition{partition}_clean_indices.npy'
    skipped_path = OUTPUT_DIR / f'partition{partition}_skipped_files.csv'
    feature_cols_path = write_feature_columns(partition, feature_cols)

    # Save metadata and clean indices regardless of tensor-saving options.
    meta_all.to_csv(meta_all_path, index=False)
    meta_clean.to_csv(meta_clean_path, index=False)
    np.save(clean_idx_path, clean_idx)
    pd.DataFrame(skipped).to_csv(skipped_path, index=False)

    if SAVE_CLEAN_TENSOR:
        X_clean = X_all[clean_idx]
        save_npz_dataset(clean_npz_path, X_clean, meta_clean)
        del X_clean
    else:
        clean_npz_path = None

    if SAVE_ALL_SENSITIVITY_TENSOR:
        save_npz_dataset(all_npz_path, X_all, meta_all)
    else:
        all_npz_path = None

    label_counts = meta_all['fl_nf_label'].value_counts().to_dict()
    flare_class_counts = meta_all['flare_class'].value_counts().to_dict()

    summary = {
        'partition': partition,
        'archive_path': str(archive_path_for_partition(partition)),
        'files_found': len(file_list),
        'processed_all': len(meta_all),
        'processed_clean': len(meta_clean),
        'skipped': len(skipped),
        'tensor_shape_all': str(tuple(X_all.shape)),
        'tensor_shape_clean': str((len(meta_clean), X_all.shape[1], X_all.shape[2])),
        'feature_count': X_all.shape[2],
        'files_with_any_interpolation': int(meta_all['has_interpolation'].sum()),
        'files_with_boundary_interpolation': int(meta_all['has_boundary_interpolation'].sum()),
        'files_with_5plus_interpolated_rows': int((meta_all['n_interpolated_rows'] >= 5).sum()),
        'total_interpolated_rows': int(meta_all['n_interpolated_rows'].sum()),
        'overall_interpolated_row_rate': float(meta_all['n_interpolated_rows'].sum() / (len(meta_all) * EXPECTED_ROWS)),
        'xrquality_degraded_instances': int(meta_all['xrquality_degraded'].sum()),
        'FL_count': int(label_counts.get('FL', 0)),
        'NF_count': int(label_counts.get('NF', 0)),
        'flare_class_counts_json': json.dumps(flare_class_counts, sort_keys=True),
        'clean_npz_path': str(clean_npz_path) if clean_npz_path else '',
        'all_sensitivity_npz_path': str(all_npz_path) if all_npz_path else '',
        'metadata_all_path': str(meta_all_path),
        'metadata_clean_path': str(meta_clean_path),
        'clean_indices_path': str(clean_idx_path),
        'feature_columns_path': str(feature_cols_path),
        'skipped_files_path': str(skipped_path),
    }

    print('\nPartition', partition, 'summary')
    print(pd.Series(summary)[[
        'processed_all', 'processed_clean', 'skipped', 'tensor_shape_clean',
        'files_with_any_interpolation', 'files_with_boundary_interpolation',
        'files_with_5plus_interpolated_rows', 'overall_interpolated_row_rate',
        'xrquality_degraded_instances', 'FL_count', 'NF_count'
    ]])

    # Free memory before the next partition.
    del X_all, meta_all, meta_clean, all_features, records, skipped
    gc.collect()

    if DELETE_EXTRACTED_AFTER_EACH_PARTITION:
        shutil.rmtree(extract_dir, ignore_errors=True)
        print(f'Partition {partition}: deleted local extraction folder to save Colab disk space.')

    return summary

## 10. Run the full build for partitions 1-5

In [ ]:
build_summaries = []

for partition in PARTITIONS_TO_BUILD:
    print('\n' + '=' * 80)
    print(f'BUILDING PARTITION {partition}')
    print('=' * 80)
    summary = build_partition(partition)
    build_summaries.append(summary)

summary_df = pd.DataFrame(build_summaries)
summary_path = OUTPUT_DIR / 'swan_sf_tensor_build_summary.csv'
summary_df.to_csv(summary_path, index=False)

print('\nSaved build summary to:', summary_path)
display(summary_df)

##  Final reminders

- The clean tensors exclude files with **5+ interpolated rows**.
- The all-file metadata still preserves every successfully processed file for sensitivity checks.
- Do not randomly split rows or files. Use the partition split or split by `HARPNUM` to avoid overlap leakage.
- Fit normalization on training partitions only, then apply that same transform to partition 4.